In [1]:
# Task 3: Data Cleaning & Preprocessing
import pandas as pd
import numpy as np

# 1. Load Dataset (Using Titanic or Messy Retail Dataset)
df_raw = pd.read_csv('https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv')
df = df_raw.copy()

# Introduce sample messy issues for demonstration (if using clean base)
df.loc[::10, 'Sex'] = ' male '
df.loc[1::15, 'Sex'] = 'MALE'

print("=== BEFORE CLEANING: Data Quality Report ===")
print(f"Shape: {df.shape}")
print(f"Duplicate Rows: {df.duplicated().sum()}")
print("\nMissing Values per Column:")
print(df.isnull().sum())
print("\nData Types:")
print(df.dtypes)

# 2. Duplicate Removal
initial_rows = len(df)
df = df.drop_duplicates()
print(f"\n[+] Removed {initial_rows - len(df)} duplicate rows.")

# 3. Standardisation (Text Formats)
df['Sex'] = df['Sex'].astype(str).str.strip().str.lower()
df['Sex'] = df['Sex'].replace({'male': 'Male', 'female': 'Female', 'm': 'Male', 'f': 'Female'})

# 4. Missing Data Handling & Justification
# Numerical: Fill 'Age' with median (robust against outliers)
df['Age'] = df['Age'].fillna(df['Age'].median())

# Categorical: Fill 'Embarked' with mode (most common port)
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

# Drop 'Cabin' column due to excessive missing data (>70%)
if 'Cabin' in df.columns:
    df = df.drop(columns=['Cabin'])

# 5. Outlier Detection & Capping using IQR Method
Q1 = df['Fare'].quantile(0.25)
Q3 = df['Fare'].quantile(0.75)
IQR = Q3 - Q1
upper_bound = Q3 + 1.5 * IQR
lower_bound = Q1 - 1.5 * IQR

# Cap outliers to upper and lower bounds
df['Fare'] = np.where(df['Fare'] > upper_bound, upper_bound, df['Fare'])
df['Fare'] = np.where(df['Fare'] < lower_bound, lower_bound, df['Fare'])

# 6. Data Type Corrections
df['PassengerId'] = df['PassengerId'].astype(str)
df['Pclass'] = df['Pclass'].astype('category')
df['Sex'] = df['Sex'].astype('category')

# 7. Before vs. After Summary Table
summary_data = {
    'Metric': ['Total Rows', 'Total Columns', 'Missing Values', 'Duplicate Rows'],
    'Before Cleaning': [df_raw.shape[0], df_raw.shape[1], df_raw.isnull().sum().sum(), df_raw.duplicated().sum()],
    'After Cleaning': [df.shape[0], df.shape[1], df.isnull().sum().sum(), df.duplicated().sum()]
}
summary_df = pd.DataFrame(summary_data)
print("\n=== BEFORE vs. AFTER SUMMARY TABLE ===")
print(summary_df.to_string(index=False))

# 8. Save Cleaned Dataset
df.to_csv('cleaned_dataset.csv', index=False)
print("\n[+] Cleaned dataset saved successfully as 'cleaned_dataset.csv'")

=== BEFORE CLEANING: Data Quality Report ===
Shape: (891, 12)
Duplicate Rows: 0

Missing Values per Column:
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

Data Types:
PassengerId      int64
Survived         int64
Pclass           int64
Name               str
Sex                str
Age            float64
SibSp            int64
Parch            int64
Ticket             str
Fare           float64
Cabin              str
Embarked           str
dtype: object

[+] Removed 0 duplicate rows.

=== BEFORE vs. AFTER SUMMARY TABLE ===
        Metric  Before Cleaning  After Cleaning
    Total Rows              891             891
 Total Columns               12              11
Missing Values              866               0
Duplicate Rows                0               0

[+] Cleaned dataset saved successful